# 🧪 Day 1 Practice Lab — Prompt Engineering Challenges
### AI/ML Bootcamp

> **Instructions:** Each section gives you a task, sample input data, and a skeleton to fill in.
> You must complete the `YOUR CODE HERE` blocks. Do NOT look at the solutions notebook until you've genuinely attempted each task.

**What this lab covers:**

| # | Task | Skill |
|---|---|---|
| 1 | Text Summarization | Length control, format control |
| 2 | Named Entity Extraction | Structured output, JSON mode |
| 3 | Tone & Style Rewriting | Role prompting, constraints |
| 4 | Classification Pipeline | Few-shot, multi-label |
| 5 | Code Review Assistant | Domain prompting, CoT |
| 6 | Data Cleaning with LLMs | Structured output, validation |
| 7 | Question Generation | Creative prompting, difficulty control |
| 8 | Multi-step Reasoning | Chain-of-thought, self-check |

**Scoring:** Each task is worth 10 points. Bonus points for optimizing token usage.

---

## ⚙️ Setup — Run this first

In [ ]:
!pip install openai tiktoken rich -q

import os, json, time
import tiktoken
from openai import OpenAI
from rich import print as rprint
from rich.table import Table
from rich.panel import Panel

# ── Add your key ──────────────────────────────
OPENAI_API_KEY = "sk-..."  
# ─────────────────────────────────────────────

client = OpenAI(api_key=OPENAI_API_KEY)
enc    = tiktoken.encoding_for_model("gpt-4o-mini")

# ── Cost tracker ─────────────────────────────
# gpt-4o-mini: $0.15 / 1M input tokens, $0.60 / 1M output tokens
COST_INPUT_PER_M  = 0.15
COST_OUTPUT_PER_M = 0.60

session_stats = {"calls": 0, "input_tokens": 0, "output_tokens": 0, "total_cost_usd": 0.0}

def llm(
    user_prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.3,
    max_tokens: int = 500,
    json_mode: bool = False,
    label: str = ""
) -> str:
    """
    Wrapper around OpenAI chat completions.
    Automatically tracks token usage and cost.
    """
    kwargs = dict(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    t0       = time.time()
    response = client.chat.completions.create(**kwargs)
    latency  = round(time.time() - t0, 2)

    in_tok  = response.usage.prompt_tokens
    out_tok = response.usage.completion_tokens
    cost    = (in_tok * COST_INPUT_PER_M + out_tok * COST_OUTPUT_PER_M) / 1_000_000

    session_stats["calls"]         += 1
    session_stats["input_tokens"]  += in_tok
    session_stats["output_tokens"] += out_tok
    session_stats["total_cost_usd"] = round(
        session_stats["total_cost_usd"] + cost, 6
    )

    tag = f"[{label}] " if label else ""
    print(f"  📊 {tag}in={in_tok} | out={out_tok} | cost=${cost:.5f} | {latency}s")
    return response.choices[0].message.content


def session_report():
    """Print a summary of all API calls made this session."""
    t = Table(title="Session Usage Report", show_header=True)
    t.add_column("Metric",  style="cyan")
    t.add_column("Value",   style="white")
    t.add_row("Total API calls",    str(session_stats["calls"]))
    t.add_row("Input tokens",       f"{session_stats['input_tokens']:,}")
    t.add_row("Output tokens",      f"{session_stats['output_tokens']:,}")
    t.add_row("Total tokens",       f"{session_stats['input_tokens'] + session_stats['output_tokens']:,}")
    t.add_row("Total cost (USD)",   f"${session_stats['total_cost_usd']:.5f}")
    rprint(t)

print("✅ Setup complete. Model: gpt-4o-mini")

---
## Task 1 — Text Summarization 📄

**Goal:** Summarize news articles in multiple formats.

**You must produce THREE summaries of the same article:**
1. A **1-sentence TL;DR** (max 25 words)
2. A **3-bullet executive summary** (each bullet max 15 words)
3. An **ELI5** — explain it like I'm 5 years old (max 60 words, no jargon)

**Graded on:**
- Each output respects the word/format constraint
- A single API call is used (put all 3 summaries in one prompt)
- Output is structured JSON with keys: `tldr`, `bullets`, `eli5`

In [ ]:
ARTICLE = """
OpenAI announced the release of GPT-5, its most capable language model to date, marking a significant
leap in AI performance across reasoning, coding, and multimodal tasks. The model reportedly scores in
the 99th percentile on standardized reasoning benchmarks and demonstrates near-human performance on
the bar exam and medical licensing tests. OpenAI CEO Sam Altman stated that GPT-5 represents the
company's first step toward what they internally call "level 3" AI — systems capable of independent
reasoning without human guidance. The model will initially be available to ChatGPT Plus and Pro
subscribers before a broader API rollout. Critics have raised concerns about the accelerating pace
of AI development and the potential societal impacts of increasingly autonomous systems. Several
AI safety researchers published an open letter urging regulators to establish clearer frameworks
before models of this capability level become widely accessible.
"""

# ── YOUR CODE HERE ───────────────────────────
# Write a system prompt and user prompt that returns
# a JSON object with keys: tldr, bullets (list of 3), eli5

system_prompt = """
# YOUR SYSTEM PROMPT
"""

user_prompt = """
# YOUR USER PROMPT
"""

result = llm(user_prompt, system_prompt, json_mode=True, label="Task1")
data   = json.loads(result)

# Print nicely
print("\nTL;DR:",   data.get("tldr"))
print("\nBullets:")
for b in data.get("bullets", []):
    print(" •", b)
print("\nELI5:",    data.get("eli5"))
# ─────────────────────────────────────────────

---
## Task 2 — Named Entity Extraction 🏷️

**Goal:** Extract structured entities from unstructured text.

**You must extract:**
- `people` — list of `{name, role}` objects
- `organizations` — list of `{name, type}` objects  
- `locations` — list of city/country names
- `dates` — list of date strings mentioned
- `monetary_values` — list of `{amount, currency, context}` objects

**Graded on:**
- All entity types are extracted correctly
- No hallucinated entities (only what's in the text)
- Valid JSON output

In [ ]:
TEXT = """
On March 15, 2024, Fatima Al-Rashid, CEO of NovaTech Solutions, announced a $47 million Series B
funding round led by Sequoia Capital. The Karachi-based startup, which also has offices in Dubai
and London, plans to use the funds to expand its AI-powered logistics platform across Southeast Asia.
Co-founder and CTO Bilal Mahmood stated the company expects to reach profitability by Q3 2025.
The deal was brokered by Goldman Sachs and is expected to close by April 30, 2024.
NovaTech had previously raised $8.2 million in a seed round in January 2022.
"""

# ── YOUR CODE HERE ───────────────────────────
system_prompt = """
# YOUR SYSTEM PROMPT
"""

user_prompt = """
# YOUR USER PROMPT
"""

result   = llm(user_prompt, system_prompt, json_mode=True, label="Task2")
entities = json.loads(result)
print(json.dumps(entities, indent=2))
# ─────────────────────────────────────────────

---
## Task 3 — Tone & Style Rewriter ✍️

**Goal:** Rewrite the same content for 4 completely different audiences.

**Rewrite the email below for:**
1. **The CEO** — executive, 3 sentences max, focus on business impact
2. **A junior intern** — friendly, simple, encouraging
3. **A lawyer** — formal, precise, no ambiguity
4. **A Twitter/X post** — punchy, engaging, max 280 chars, include relevant hashtags

**Graded on:**
- Tone is clearly different for each audience
- Content/facts are preserved across all rewrites
- Twitter version is actually under 280 characters (verify this in code!)

In [ ]:
ORIGINAL_EMAIL = """
Hey team,

Just a heads up — the deployment we did yesterday introduced a bug that's causing about 3% of
users to see a blank screen on the dashboard. We've identified the issue (a null pointer in the
analytics module) and a fix is already in review. ETA to production is roughly 2 hours.
No data loss, no security risk. About 1,200 users are affected.

Will update once it's resolved.
— Dev Team
"""

# ── YOUR CODE HERE ───────────────────────────
# Hint: You can do this in one API call using JSON mode
# OR in 4 separate calls. Which is cheaper? Try both!

system_prompt = """
# YOUR SYSTEM PROMPT
"""

user_prompt = """
# YOUR USER PROMPT
"""

result  = llm(user_prompt, system_prompt, json_mode=True, label="Task3")
rewrites = json.loads(result)

for audience, text in rewrites.items():
    print(f"\n{'='*50}")
    print(f"Audience: {audience.upper()}")
    print(text)

# Verify Twitter length
twitter_text = rewrites.get("twitter", "")
char_count   = len(twitter_text)
status       = "✅ Within limit" if char_count <= 280 else f"❌ Too long by {char_count - 280} chars"
print(f"\nTwitter char count: {char_count}/280 — {status}")
# ─────────────────────────────────────────────

---
## Task 4 — Multi-Label Classification Pipeline 🏷️

**Goal:** Build a classifier that categorizes support tickets into multiple labels simultaneously.

**Each ticket can have:**
- `category`: one of `[billing, technical, shipping, account, feature_request, complaint, other]`
- `urgency`: one of `[low, medium, high, critical]`
- `sentiment`: one of `[positive, neutral, frustrated, angry]`
- `needs_human`: `true` or `false`
- `suggested_team`: one of `[billing_team, tech_support, logistics, customer_success, product_team]`

**Graded on:**
- Consistent, sensible classifications across all 6 tickets
- Use **few-shot examples** in your prompt (at least 2)
- Output a summary table at the end

In [ ]:
TICKETS = [
    "I was charged twice for my subscription this month. Please refund immediately!",
    "The mobile app keeps crashing when I try to upload images larger than 5MB.",
    "My package was supposed to arrive Tuesday, it's now Friday. Where is it?",
    "Would love a dark mode option for the dashboard, the white is really harsh at night.",
    "I forgot my password and the reset email never arrived. I've tried 5 times.",
    "Everything is working great! Just wanted to say the new UI update looks amazing.",
]

# ── YOUR CODE HERE ───────────────────────────
# Step 1: Write a system prompt with at least 2 few-shot examples
# Step 2: Loop over each ticket and classify it
# Step 3: Print a summary table using rich

system_prompt = """
# YOUR SYSTEM PROMPT WITH FEW-SHOT EXAMPLES
"""

classifications = []

for i, ticket in enumerate(TICKETS):
    user_prompt = f"Classify this ticket:\n{ticket}"
    result = llm(user_prompt, system_prompt, json_mode=True, temperature=0, label=f"Task4-t{i+1}")
    classifications.append({"ticket": ticket[:50] + "...", **json.loads(result)})

# Print results table
table = Table(title="Ticket Classifications", show_header=True)
# YOUR TABLE CODE HERE
rprint(table)
# ─────────────────────────────────────────────

---
## Task 5 — Code Review Assistant 💻

**Goal:** Build a code reviewer that gives actionable, structured feedback.

**The reviewer must output:**
- `overall_score`: integer 1-10
- `summary`: one-sentence verdict
- `issues`: list of `{severity: critical|major|minor, line_hint: str, problem: str, fix: str}`
- `positives`: list of things done well
- `refactored_snippet`: the corrected version of the worst issue only

**Graded on:**
- Correctly identifies the real bugs (there are 4 issues in the code below)
- `refactored_snippet` actually fixes the critical issue
- Uses chain-of-thought reasoning before producing output

In [ ]:
CODE_TO_REVIEW = """
import hashlib

def authenticate_user(username, password, users_db):
    # Check if user exists
    if username in users_db:
        stored_password = users_db[username]
        if stored_password == password:  # Direct comparison
            print(f"User {username} logged in with password {password}")
            return True
    return False

def get_user_data(user_id, database):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = database.execute(query)
    return result

def process_items(items):
    total = 0
    for i in range(len(items)):
        total = total + items[i]
    return total / len(items)
"""

# ── YOUR CODE HERE ───────────────────────────
# Hint: use chain-of-thought — ask the model to "think first"
# before producing the JSON. You can use a two-call approach:
# Call 1: reason about the code (free text)
# Call 2: produce structured JSON based on the reasoning

system_prompt = """
# YOUR SYSTEM PROMPT
"""

user_prompt = """
# YOUR USER PROMPT
"""

result  = llm(user_prompt, system_prompt, json_mode=True, max_tokens=800, label="Task5")
review  = json.loads(result)

print(f"\nScore: {review.get('overall_score')}/10")
print(f"Summary: {review.get('summary')}")
print("\nIssues:")
for issue in review.get("issues", []):
    print(f"  [{issue.get('severity', '').upper()}] {issue.get('problem')}")
    print(f"     Fix: {issue.get('fix')}")
print("\nRefactored snippet:")
print(review.get("refactored_snippet"))
# ─────────────────────────────────────────────

---
## Task 6 — Messy Data Cleaning 🧹

**Goal:** Use an LLM to normalize and clean inconsistent real-world data.

**Clean the following fields:**
- `name` → proper case, remove extra spaces
- `phone` → normalize to `+1-XXX-XXX-XXXX` format (assume US numbers)
- `email` → lowercase, flag invalid emails with `null`
- `company` → normalize (e.g. "intl" → "International", "inc" → "Inc.")
- `joined_date` → normalize to `YYYY-MM-DD` format
- `is_valid_record`: `true` only if name, email, and phone are all present and valid

**Graded on:**
- All 6 records are cleaned correctly
- Invalid/missing data is handled gracefully (not hallucinated)
- Output is a JSON array of cleaned records

In [ ]:
MESSY_DATA = [
    {"name": "  john   SMITH ",    "phone": "(555) 123-4567",    "email": "John.Smith@Gmail.COM",   "company": "acme corp intl",      "joined_date": "Jan 5, 2023"},
    {"name": "sara ali",           "phone": "555.987.6543",       "email": "sara@",                  "company": "TechStartup Inc",      "joined_date": "2023/08/22"},
    {"name": "DR. MICHAEL  JONES", "phone": "+1 (800) 555-0199",  "email": "m.jones@hospital.org",   "company": "city hospital llc",    "joined_date": "12-01-2022"},
    {"name": "",                   "phone": "not provided",        "email": "unknown@test.com",        "company": "freelance",            "joined_date": "March 2024"},
    {"name": "Yuki Tanaka",        "phone": "5551234",             "email": "yuki.t@company.co.jp",   "company": "Tanaka Enterprises co", "joined_date": "2024-01-15"},
    {"name": "fatima   AL rashid", "phone": "(+1) 212-555-7890",  "email": "FATIMA@NOVATECH.IO",     "company": "novatech solutions ltd", "joined_date": "5th Sep 2021"},
]

# ── YOUR CODE HERE ───────────────────────────
# Tip: Send all records at once in a single call
# Include the schema rules clearly in your prompt

system_prompt = """
# YOUR SYSTEM PROMPT
"""

user_prompt = f"""
# YOUR USER PROMPT
Messy data: {json.dumps(MESSY_DATA, indent=2)}
"""

result  = llm(user_prompt, system_prompt, json_mode=True, max_tokens=800, label="Task6")
cleaned = json.loads(result)
print(json.dumps(cleaned, indent=2))
# ─────────────────────────────────────────────

---
## Task 7 — Quiz Question Generator 🎓

**Goal:** Auto-generate a quiz from a passage of text.

**Generate 5 multiple-choice questions where:**
- Each question has exactly 4 options (A, B, C, D)
- One correct answer and three plausible but wrong distractors
- Questions span 3 difficulty levels: `easy` (2), `medium` (2), `hard` (1)
- Each question includes a `explanation` of why the answer is correct

**Graded on:**
- Questions are actually answerable from the passage (no outside knowledge needed)
- Distractors are plausible (not obviously wrong)
- Difficulty levels feel genuinely different
- Code auto-runs a mock quiz and scores it

In [ ]:
PASSAGE = """
The transformer architecture, introduced in the 2017 paper "Attention is All You Need" by Vaswani
et al., revolutionized natural language processing by replacing recurrent neural networks with a
purely attention-based mechanism. The core innovation is the self-attention mechanism, which allows
each token in a sequence to attend to every other token, capturing long-range dependencies more
effectively than RNNs. Transformers consist of an encoder-decoder structure, though many modern
models use only the encoder (e.g., BERT) or only the decoder (e.g., GPT). Training requires massive
datasets and compute, but the architecture scales well — performance tends to improve predictably
with more data and parameters, a phenomenon known as scaling laws. The attention mechanism's
complexity is O(n²) with sequence length, which poses challenges for very long documents.
"""

# ── YOUR CODE HERE ───────────────────────────
system_prompt = """
# YOUR SYSTEM PROMPT
"""

user_prompt = """
# YOUR USER PROMPT
"""

result = llm(user_prompt, system_prompt, json_mode=True, max_tokens=1200, label="Task7")
quiz   = json.loads(result)

# Auto-run the quiz (simulate a student answering randomly)
import random
score = 0
questions = quiz.get("questions", [])

for i, q in enumerate(questions):
    student_answer = random.choice(["A", "B", "C", "D"])  # random student
    correct        = q.get("correct_answer")
    is_right       = student_answer == correct
    if is_right: score += 1
    
    print(f"\nQ{i+1} [{q.get('difficulty')}]: {q.get('question')}")
    for opt_key, opt_val in q.get("options", {}).items():
        marker = "✅" if opt_key == correct else "  "
        chosen = " ← your answer" if opt_key == student_answer else ""
        print(f"  {marker} {opt_key}) {opt_val}{chosen}")
    print(f"  📖 {q.get('explanation')}")

print(f"\n🎯 Score: {score}/{len(questions)}")
# ─────────────────────────────────────────────

---
## Task 8 — Multi-step Reasoning with Self-Check 🧠

**Goal:** Implement a "reason → answer → verify" pipeline for complex questions.

**The pipeline has 3 steps:**
1. **Reasoning call** — Ask the model to think through the problem step by step (free text, no JSON)
2. **Answer call** — Based on its reasoning, produce a final structured answer
3. **Verification call** — A *second independent call* checks the answer and flags any errors

**Graded on:**
- Each step is a separate API call (3 calls total per problem)
- The verification step sometimes catches an error (test with a tricky question)
- Total token usage and cost are printed after all 3 calls
- Implement this as a reusable `reason_and_verify(question)` function

In [ ]:
QUESTIONS = [
    "A snail is at the bottom of a 10-foot well. Each day it climbs 3 feet, but each night it slips back 2 feet. How many days does it take to reach the top?",
    "If you have a 3-gallon jug and a 5-gallon jug, how do you measure exactly 4 gallons of water?",
    "A bat and a ball cost $1.10 together. The bat costs $1.00 more than the ball. How much does the ball cost?",
]

# ── YOUR CODE HERE ───────────────────────────
def reason_and_verify(question: str) -> dict:
    """
    3-step pipeline:
    1. reason  → free text chain-of-thought
    2. answer  → structured JSON {answer, confidence, assumptions}
    3. verify  → {is_correct: bool, issues: str, corrected_answer: str or null}
    Returns a dict with all three outputs and combined token stats.
    """
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    
    # ── Step 1: Reason ───────────────────────
    reasoning = llm(
        # YOUR PROMPT
        label="Task8-reason"
    )
    
    # ── Step 2: Answer ───────────────────────
    answer_json = llm(
        # YOUR PROMPT (include the reasoning above)
        json_mode=True,
        label="Task8-answer"
    )
    answer = json.loads(answer_json)
    
    # ── Step 3: Verify ───────────────────────
    verification_json = llm(
        # YOUR PROMPT (independent check)
        json_mode=True,
        label="Task8-verify"
    )
    verification = json.loads(verification_json)
    
    print(f"\n  Reasoning:\n{reasoning[:300]}...")
    print(f"  Answer: {answer.get('answer')} (confidence: {answer.get('confidence')})")
    print(f"  Verification: {'✅ Correct' if verification.get('is_correct') else '❌ Issues found'}")
    if not verification.get('is_correct'):
        print(f"  Issues: {verification.get('issues')}")
        print(f"  Corrected: {verification.get('corrected_answer')}")
    
    return {"reasoning": reasoning, "answer": answer, "verification": verification}


# Run on all questions
for q in QUESTIONS:
    reason_and_verify(q)
# ─────────────────────────────────────────────

---
## 📊 Final Session Report

Run this at the end to see your total token usage and cost across all tasks.

In [ ]:
session_report()

print("\n💡 Reflection questions:")
print("  1. Which task used the most tokens? Why?")
print("  2. Which task could be made cheaper with a better prompt?")
print("  3. Which task benefited most from few-shot examples?")
print("  4. Where did the model make mistakes? What prompt change would fix it?")